# Gradient Descent — run the loop yourself

A short companion to the lesson [*Gradient Descent*](https://lms-p-45c03.web.app/topics/math-infra/gradient-descent/).
Run the training loop on a simple loss surface and **chart** what the learning rate does — converge,
crawl, or blow up to infinity. **Runs on CPU**: `numpy` + `matplotlib`, no calculus, no GPU.

## 1. The loop
A model's "loss surface" here is a simple bowl `f(w) = ½(w₀² + 8·w₁²)` — elongated, so the steep
axis is where things go wrong. Gradient descent: step downhill, scaled by the learning rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

K = 8.0                                   # steep axis -> stability needs lr < 2/K = 0.25
START = np.array([-0.9, 0.8])
loss = lambda w: 0.5 * (w[0]**2 + K * w[1]**2)
grad = lambda w: np.array([w[0], K * w[1]])

def descend(lr, mu=0.0, steps=40):
    w, v = START.copy(), np.zeros(2)
    path, L = [w.copy()], [loss(w)]
    for _ in range(steps):
        v = mu * v - lr * grad(w); w = w + v
        path.append(w.copy()); L.append(loss(w))
        if not np.isfinite(w).all() or np.hypot(*w) > 5: break   # diverged
    return np.array(path), np.array(L)

for lr in [0.05, 0.12, 0.24, 0.30]:
    _, L = descend(lr)
    tag = "DIVERGED" if (not np.isfinite(L[-1])) or L[-1] > L[0] else "ok"
    print(f"lr={lr}: {len(L)-1:>2} steps, final loss {L[-1]:.2e}  [{tag}]")

## 2. Loss vs step — the stability band
Plot the loss each step for several learning rates. Small = crawls; in-band = drops fast;
too big (0.30 > 0.25) = **explodes** — the NaN blow-up, an unstable control loop.

In [ ]:
plt.figure(figsize=(8, 4.5))
for lr, c in [(0.05, "#94a3b8"), (0.12, "#0d9488"), (0.24, "#2563eb"), (0.30, "#dc2626")]:
    _, L = descend(lr)
    Lc = np.clip(np.nan_to_num(L, posinf=1e8), 1e-12, 1e8)
    plt.plot(Lc, label=f"lr = {lr}", lw=2.2, color=c, marker="o", ms=3)
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log)")
plt.title("Learning rate sets the regime: crawl / converge / diverge")
plt.legend(); plt.grid(alpha=.25, which="both"); plt.tight_layout(); plt.show()

## 3. The path on the loss surface
Same loop, drawn on the contours. At `lr=0.24` the path **zig-zags** across the steep valley but
still reaches the minimum; at `lr=0.30` it overshoots and **flies off**.

In [ ]:
xs = np.linspace(-1.2, 1.2, 140); X, Y = np.meshgrid(xs, xs); Z = 0.5 * (X**2 + K * Y**2)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (lr, title) in zip(axes, [(0.24, "lr=0.24 — zig-zags but converges"), (0.30, "lr=0.30 — diverges")]):
    ax.contour(X, Y, Z, levels=[0.1, 0.4, 1, 2, 4], colors="#94a3b8", linewidths=1)
    p, _ = descend(lr)
    ax.plot(p[:, 0], p[:, 1], "-o", ms=3, color="#dc2626" if lr > 0.25 else "#2563eb")
    ax.plot(0, 0, "*", ms=14, color="#0d9488")           # the minimum
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

## Put it together
1. **§2:** which learning rate converges fastest, and which one blows up? Where's the stability edge (hint: 2/K = 0.25)?
2. **§3:** at `lr=0.24` the path zig-zags. What in the loss surface causes that, and what would smooth it (try adding `mu=0.8` to `descend`)?
3. A run's loss goes to `NaN` after a few steps. From these charts, what's your one-line diagnosis — and what would you tell the team that owns the run?

Back to the lesson → [Gradient Descent](https://lms-p-45c03.web.app/topics/math-infra/gradient-descent/)